# AethyxLM — Production training on Kaggle (2× T4)

This notebook resumes the current **31.2M-parameter, 32K-tokenizer-v2** mixed-dataset run with true two-GPU Distributed Data Parallel (DDP). It is compatible with Kaggle's native notebook UI and with a Kaggle Jupyter Server connected to VS Code or Colab.

Before running:

1. Select Kaggle's **GPU T4 ×2** accelerator.
2. Attach a private Kaggle Dataset containing every active `*.bin` listed by `configs/datasets.json`. The notebook memory-maps these files directly from `/kaggle/input`, so it does not duplicate them into limited working storage.
3. Attach the latest checkpoint or a previous notebook output. Numbered checkpoints are preferred automatically; `checkpoint_latest.pt` is the fallback.
4. Every numbered checkpoint is automatically uploaded as a new version of the private `aethyx/aethyxlm-live-checkpoints` Kaggle Dataset before training continues.

The tokenizer is loaded from the repository and is **not retrained**. The global effective batch remains 32 sequences: `16 per GPU × 2 GPUs × 1 accumulation step`. Checkpoints are saved every 1,000 optimizer steps.

> `/kaggle/working` survives while the VM is alive, including Jupyter Server reconnects, but not a recycled Kaggle session. Save a notebook version with outputs or publish the output checkpoints as a private Kaggle Dataset before the session expires.

In [ ]:
# 1. Prepare the Kaggle workspace and repository
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, time

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('This notebook must run inside a Kaggle environment.')

REPO_URL = 'https://github.com/aethyx-ai/AethyxLM.git'
WORK_ROOT = Path('/kaggle/working/aethyxlm')
REPO_ROOT = WORK_ROOT / 'repo'
PROJECT_ROOT = REPO_ROOT / 'AethyxLM'
OUTPUT_ROOT = Path('/kaggle/working/aethyxlm_output')
OUTPUT_CHECKPOINTS = OUTPUT_ROOT / 'checkpoints'
OUTPUT_LOGS = OUTPUT_ROOT / 'logs'
OUTPUT_CONFIGS = OUTPUT_ROOT / 'configs'
OUTPUT_MILESTONES = OUTPUT_CHECKPOINTS / 'milestones'
for path in (WORK_ROOT, OUTPUT_CHECKPOINTS, OUTPUT_LOGS, OUTPUT_CONFIGS, OUTPUT_MILESTONES):
    path.mkdir(parents=True, exist_ok=True)

# Set this to an exact attached checkpoint path to override automatic detection.
RESUME_CHECKPOINT = None
REQUIRE_RESUME = True  # prevents accidentally restarting the existing run from step 0

# These variables are inherited by torchrun's two worker processes.
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git checkout; remove or rename it first.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

if not (PROJECT_ROOT / 'train.py').is_file():
    raise FileNotFoundError(f'Expected the training project at {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep Kaggle's CUDA-enabled PyTorch build; install only project-side dependencies.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tokenizers>=0.13.0', 'datasets>=2.14.0', 'tensorboard>=2.14.0',
    'tqdm>=4.65.0', 'pyyaml>=6.0'
], check=True)
print(f'[OK] Project: {PROJECT_ROOT}')
print(f'[OK] Runtime outputs: {OUTPUT_ROOT}')

In [ ]:
# 2. Verify that Kaggle exposed both T4 GPUs and NCCL
import torch

print(f'PyTorch: {torch.__version__}; CUDA build: {torch.version.cuda}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select GPU T4 x2 in Kaggle notebook settings.')
if torch.cuda.device_count() != 2:
    raise RuntimeError(f'Expected exactly 2 GPUs, but PyTorch sees {torch.cuda.device_count()}. Select GPU T4 x2.')
if not torch.distributed.is_nccl_available():
    raise RuntimeError('This PyTorch build does not provide NCCL, which is required for two-GPU DDP.')

gpu_names = []
for index in range(2):
    props = torch.cuda.get_device_properties(index)
    gpu_names.append(props.name)
    print(f'GPU {index}: {props.name} | {props.total_memory / 2**30:.1f} GiB')
if not all('T4' in name for name in gpu_names):
    print('[WARN] Two CUDA GPUs are available, but they are not both reported as T4s.')
print('[OK] Dual-GPU DDP prerequisites are available.')

In [ ]:
# 3. Locate prepared token binaries in attached Kaggle Datasets
source_registry_path = PROJECT_ROOT / 'configs' / 'datasets.json'
source_registry = json.loads(source_registry_path.read_text(encoding='utf-8'))
input_root = Path('/kaggle/input')

def find_attached_file(relative_path):
    filename = Path(relative_path).name
    candidates = [path for path in input_root.rglob(filename) if path.is_file() and path.stat().st_size > 0]
    if not candidates:
        return None
    # Prefer the largest candidate if multiple attached datasets contain the same name.
    selected = max(candidates, key=lambda path: path.stat().st_size)
    if len(candidates) > 1:
        print(f'[WARN] Multiple copies of {filename}; using {selected}')
    return selected

runtime_registry = {}
missing = []
selected_files = {}
for dataset_name, entry in source_registry.items():
    runtime_entry = {'weight': entry.get('weight', 1.0)}
    for split in ('train', 'val'):
        attached = find_attached_file(entry[split])
        if attached is None:
            missing.append(Path(entry[split]).name)
        else:
            runtime_entry[split] = str(attached)
            selected_files[str(attached)] = attached.stat().st_size
    runtime_registry[dataset_name] = runtime_entry

if missing:
    preview = '\n'.join(f'  - {name}' for name in sorted(set(missing)))
    raise FileNotFoundError('Attach a Kaggle Dataset containing these prepared binaries:\n' + preview)

runtime_registry_path = OUTPUT_CONFIGS / 'datasets_kaggle.json'
runtime_registry_path.write_text(json.dumps(runtime_registry, indent=2) + '\n', encoding='utf-8')
total_bytes = sum(selected_files.values())
print(f'[OK] Located {len(selected_files)} unique binaries ({total_bytes / 2**30:.2f} GiB).')
print('[OK] They will be memory-mapped directly from /kaggle/input; no local dataset copy was made.')
print(f'[OK] Tokenizer v2: {PROJECT_ROOT / "tokenizer/tokenizer.json"}')

In [ ]:
# 4. Build a dual-T4 production configuration
base_config_path = PROJECT_ROOT / 'configs' / 'train_config_modern.json'
config = json.loads(base_config_path.read_text(encoding='utf-8'))

# Match the previous single-T4 global batch of 32 while removing accumulation overhead.
per_gpu_batch = 16
world_size = 2
grad_accum_steps = 1
config['training'].update({
    'batch_size': per_gpu_batch,
    'grad_accum_steps': grad_accum_steps,
    'use_amp': True,
    'amp_dtype': 'float16',
    'fused_optimizer': True,
    'torch_compile': False,
    'eval_batches': 25,  # 25 per rank = 50 total, matching the single-GPU evaluation budget
})
config['data'].update({
    'datasets_file': str(runtime_registry_path),
    'batch_size': per_gpu_batch,
    'num_workers': 2,
    'shuffle': False,
})
config['checkpoint'].update({
    'checkpoint_dir': str(OUTPUT_CHECKPOINTS),
    'milestone_dir': str(OUTPUT_MILESTONES),
    'milestone_interval': 10000,
    'metrics_file': str(OUTPUT_LOGS / 'metrics.jsonl'),
    'log_dir': str(OUTPUT_LOGS),
    'tensorboard_dir': str(OUTPUT_LOGS / 'tensorboard'),
    'save_interval': 1000,
    'log_interval': 50,
    'backup': {
        'enabled': True,
        'provider': 'kaggle_dataset',
        'handle': 'aethyx/aethyxlm-live-checkpoints',
        'required': True,
        'retries': 3,
    },
})

kaggle_config_path = OUTPUT_CONFIGS / 'train_config_kaggle_2xt4.json'
kaggle_config_path.write_text(json.dumps(config, indent=2) + '\n', encoding='utf-8')
effective_batch = per_gpu_batch * world_size * grad_accum_steps
print(f'[OK] Config: {kaggle_config_path}')
print(f'Global effective batch: {per_gpu_batch} × {world_size} × {grad_accum_steps} = {effective_batch}')
print(f'Max step: {config["training"]["max_steps"]:,}; save every {config["checkpoint"]["save_interval"]:,} steps')

# Validate model, tokenizer, registry, and every binary before launching two workers.
subprocess.run([
    sys.executable, 'scripts/check_training_readiness.py', '--config', str(kaggle_config_path)
], cwd=PROJECT_ROOT, check=True)

In [ ]:
# 5. Select the checkpoint to resume
import gc, re

def valid_checkpoint(path):
    return path.is_file() and path.stat().st_size > 10 * 2**20

def discover_checkpoints():
    roots = [OUTPUT_CHECKPOINTS, Path('/kaggle/input'), PROJECT_ROOT / 'checkpoints']
    found = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for pattern in ('checkpoint_step_*.pt', 'checkpoint_latest.pt', 'checkpoint_best.pt'):
            for path in root.rglob(pattern):
                resolved = path.resolve()
                if resolved not in seen and valid_checkpoint(path):
                    found.append(path)
                    seen.add(resolved)
    return found

if RESUME_CHECKPOINT is not None:
    resume_path = Path(RESUME_CHECKPOINT).expanduser()
    if not valid_checkpoint(resume_path):
        raise FileNotFoundError(f'Invalid RESUME_CHECKPOINT: {resume_path}')
else:
    checkpoint_candidates = discover_checkpoints()
    numbered = []
    for path in checkpoint_candidates:
        match = re.fullmatch(r'checkpoint_step_(\d+)\.pt', path.name)
        if match:
            numbered.append((int(match.group(1)), path))
    if numbered:
        resume_path = max(numbered, key=lambda item: item[0])[1]
    else:
        latest = [path for path in checkpoint_candidates if path.name == 'checkpoint_latest.pt']
        best = [path for path in checkpoint_candidates if path.name == 'checkpoint_best.pt']
        resume_path = (latest or best or [None])[0]

if resume_path is None:
    if REQUIRE_RESUME:
        raise FileNotFoundError('No checkpoint found. Attach the current checkpoint dataset, or set REQUIRE_RESUME=False.')
    resume_args = []
    print('[WARN] No checkpoint found; training will start from step 0.')
else:
    # Read metadata once so an accidental older resume is visible before training starts.
    checkpoint_metadata = torch.load(resume_path, map_location='cpu', weights_only=False)
    print(f'[OK] Resume checkpoint: {resume_path}')
    print(f'Step: {checkpoint_metadata.get("step", "unknown")}; tokens seen: {checkpoint_metadata.get("tokens_seen", "unknown")}')
    resume_args = ['--resume', str(resume_path)]
    del checkpoint_metadata
    gc.collect()

In [ ]:
# 6. Launch two DDP workers and stream logs to this Jupyter client
command = [
    sys.executable, '-m', 'torch.distributed.run',
    '--standalone', '--nproc_per_node=2',
    'train.py', '--config', str(kaggle_config_path), '--ddp',
] + resume_args

print('Running:', ' '.join(command))
print(f'Checkpoints: {OUTPUT_CHECKPOINTS}')
print('Keep this cell running; reconnecting VS Code/Colab does not stop the Kaggle kernel.')
started = time.time()
process = None
return_code = None
try:
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, env=os.environ.copy(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    print('\nInterrupt received; asking the trainer to save a graceful-shutdown checkpoint...')
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
        try:
            return_code = process.wait(timeout=180)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            return_code = process.wait(timeout=30)
finally:
    elapsed_hours = (time.time() - started) / 3600
    print(f'Process exit code: {return_code}; elapsed: {elapsed_hours:.2f} hours')

if return_code not in (0, 130, -signal.SIGINT):
    raise RuntimeError(f'Training exited unexpectedly with code {return_code}. Inspect the final log lines above.')

In [ ]:
# 7. Inspect outputs and persistence status
checkpoints = sorted(
    OUTPUT_CHECKPOINTS.glob('*.pt'),
    key=lambda path: path.stat().st_mtime,
)
if not checkpoints:
    print('No output checkpoint has been written yet.')
else:
    print('Available output checkpoints:')
    for path in checkpoints:
        print(f'  {path.name:32s} {path.stat().st_size / 2**20:8.1f} MiB')

print(f'\nMetrics: {OUTPUT_LOGS / "metrics.jsonl"}')
print(f'All outputs: {OUTPUT_ROOT}')
print('IMPORTANT: before ending/restarting the Kaggle VM, save a notebook version with outputs')
print('or create/update a private Kaggle Dataset from /kaggle/working/aethyxlm_output.')
print('Attach that saved output on the next session; Cell 5 will resume from its highest numbered checkpoint.')